# 06 Subhalo comparison

Collects the test scores of the subhalo runs into tables (four-class Matthews correlation coefficient (MCC) and void scores by run and spacing), compares void precision and recall across runs, and checks that the FiLM (feature-wise linear modulation, spacing-conditioned) models actually use the spacing they are given: each is scored on the spacing-10 field while being told spacings from 3 to 10. If the scores barely change, the conditioning does nothing.

Section 0 re-scores older runs with a wider margin between prediction tiles. No training here. Runtime: **L4 GPU** with **High-RAM** (the sweep predicts the full box several times).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Re-score finished runs

Scores every finished subhalo run the same way: a 32-cell margin between prediction tiles, predicted classes by highest score (argmax), the void-vs-rest scores, void precision–recall curves and void-score calibration, and, for comparison only, the void scores with a void threshold chosen on the validation slab. Results go to `test_scores_margin32.json` and `test_figure_margin32.png` in each run folder (the original scores are kept). About 3–5 minutes per run; runs already done, and runs still training, are skipped.

In [ ]:
import json
from pathlib import Path
from deepvoid.train import evaluate_run

root = Path('/content/drive/MyDrive/deepvoid_runs')
RUNS = ['subhalo_A_scratch_L10', 'subhalo_B_finetune_L10', 'subhalo_C_mixed',
        'subhalo_A_smooth2ch_L10', 'subhalo_B_smooth2ch_L10', 'subhalo_C_mixed_smooth2ch',
        'subhalo_A_smoothonly_L10', 'subhalo_C_mixed_smooth2ch_seed13',
        'subhalo_C_mixed_smooth2ch_seed14', 'subhalo_C_mixed_smooth2ch_depth5',
        'subhalo_C_mixed_smooth2ch_voidloss', 'subhalo_C_mixed_smooth2ch_deepsup']
for run_name in RUNS:
    out = root / run_name
    path = out / 'test_scores_margin32.json'
    first = next(iter(json.loads(path.read_text()).values())) if path.exists() else {}
    done = 'void_curve' in first and 'tuned' in first
    finished = (out / 'test_scores.json').exists()   # written when training ends
    if done:
        print('already done:', run_name)
    elif not finished:
        print('not finished yet, skipped:', run_name)
    else:
        print('re-scoring', run_name)
        evaluate_run(out, margin=32, void_threshold='tune')


## 1. Test scores by run and spacing

Three tables: four-class MCC; void MCC (void against the rest), with the thresholded value in brackets for comparison; and void precision / recall. All headline numbers use the highest-score class (argmax).

In [ ]:
import pandas as pd

def load_scores(run_name):
    for name in ('test_scores_margin32.json', 'test_scores.json'):
        path = root / run_name / name
        if path.exists():
            return json.loads(path.read_text())
    return None

def cell(value, fmt='{:.3f}'):
    return '' if value is None else fmt.format(value)

mcc, void, purity = {}, {}, {}
for run_name in RUNS:
    scores = load_scores(run_name)
    if scores is None:
        print('not finished:', run_name)
        continue
    for name, s in scores.items():
        L = name.rsplit('/', 1)[-1]
        t = s.get('tuned')
        mcc.setdefault(run_name, {})[L] = s['mcc']
        void.setdefault(run_name, {})[L] = cell(s.get('void_mcc')) + (
            f" ({t['void_mcc']:.3f} @ {t['void_threshold']:.2f})" if t else '')
        purity.setdefault(run_name, {})[L] = (f"{s['void_precision']:.3f} / {s['void_recall']:.3f}"
                                              if 'void_precision' in s else '')

cols = ['L3', 'L5', 'L7', 'L10']
print('Four-class MCC (argmax)')
print(pd.DataFrame(mcc).T.reindex(columns=cols).round(3).to_string())
print('\nVoid MCC, argmax (in brackets: with a void threshold chosen on the validation slab @ threshold)')
print(pd.DataFrame(void).T.reindex(columns=cols).to_string())
print('\nVoid precision / recall (argmax)')
print(pd.DataFrame(purity).T.reindex(columns=cols).to_string())
tables = {'mcc': mcc, 'void_mcc_argmax_and_tuned': void, 'void_precision_recall': purity}

from deepvoid.report import Report
rep = Report(root / 'subhalo_comparison')
rep.record(**tables)


### Run-to-run spread

The same config trained with different seeds (`subhalo_C_mixed_smooth2ch` is seed 12). Differences between configs smaller than this spread cannot be told apart.

In [ ]:
import numpy as np
seed_runs = ['subhalo_C_mixed_smooth2ch', 'subhalo_C_mixed_smooth2ch_seed13',
             'subhalo_C_mixed_smooth2ch_seed14']
found = {r: load_scores(r) for r in seed_runs}
found = {r: s for r, s in found.items() if s and 'void_mcc' in next(iter(s.values()))}
if len(found) < 2:
    print('need at least two finished seeds; found', list(found))
else:
    spread = {}
    for metric in ('void_mcc', 'mcc'):
        for L in ('L3', 'L5', 'L7', 'L10'):
            vals = [s[f'tng300-3/subhalos/{L}'][metric] for s in found.values()]
            spread.setdefault(metric, {})[L] = f'{np.mean(vals):.3f} ± {np.std(vals, ddof=1):.3f}'
    print(f'mean ± standard deviation over {len(found)} seeds')
    print(pd.DataFrame(spread).T.to_string())
    rep.record(seed_spread=spread)

## 2. Void precision against recall

For void against everything else on the test slab: precision (purity: the fraction of voxels called void that are void) against recall (completeness: the fraction of void voxels called void), traced out by moving the void-score threshold from 0 to 1. A curve further up and to the right is better at every threshold. The dots mark each run's actual predictions (highest-score class). The legend gives each curve's average precision (AP), the area under it: 1 is perfect, and a model that ignores its input scores the void fraction.

The table gives AP and two points read off each curve: the best precision at recall ≥ 0.9, and the best recall at precision ≥ 0.9. Needs the re-scored files from section 0.

In [ ]:
from deepvoid.evaluate import average_precision, precision_at_recall, recall_at_precision
from deepvoid.figures import void_pr_figure

curves, points, readout = {}, {}, {}
for run_name in RUNS:
    scores = load_scores(run_name)
    if not scores:
        continue
    for name, s in scores.items():
        c = s.get('void_curve', {}).get('test')
        if c is None:
            continue
        L = name.rsplit('/', 1)[-1]
        curves.setdefault(run_name, {})[L] = c
        points.setdefault(run_name, {})[L] = (s['void_recall'], s['void_precision'])
        readout.setdefault(('average precision', run_name), {})[L] = average_precision(c)
        readout.setdefault(('precision at recall >= 0.9', run_name), {})[L] = precision_at_recall(c, 0.9)
        readout.setdefault(('recall at precision >= 0.9', run_name), {})[L] = recall_at_precision(c, 0.9)

fig = void_pr_figure(curves, points)
rep.show(fig, 'void_precision_recall')
table = pd.DataFrame(readout).T.reindex(columns=['L3', 'L5', 'L7', 'L10']).round(3)
print(table.to_string())
rep.record(void_pr_readout={f'{k[0]} | {k[1]}': v for k, v in readout.items()})

## 3. Spacing sweep for the FiLM runs

Takes a few minutes per run. The true spacing is 10.

In [ ]:
from deepvoid.train import spacing_sweep

told = [3, 5, 7, 10]
sweeps = {}
for run_name in ['subhalo_C_film', 'subhalo_C_film_curriculum']:
    if (root / run_name / 'best.keras').exists():
        result = spacing_sweep(root / run_name, 'tng300-3/subhalos/L10', told)
        sweeps[run_name] = {s: r['mcc'] for s, r in result.items()}
print(pd.DataFrame(sweeps).round(3))
rep.record(spacing_sweep_mcc=sweeps)

All tables are also saved in `deepvoid_runs/subhalo_comparison/results.json`, and the figures next to it.